# Pollinator YOLO - thesis run

Thesis-specific YOLO training + evaluation pipeline. Run this instead of
`pollinator_pipeline.ipynb` for the iteration that produces the numbers that go into the
thesis results section. The crop-based classifiers (binary, group) are intentionally not
trained here - those branches are out of scope for the YOLO results.

What this notebook adds on top of the production pipeline:

- Three-class training (`fly`, `butterfly`, `other`); bumblebee excluded (14 instances total).
- Plot-stratified train/val/test re-split so every camera plot is represented in every split.
- `slicer_stats.json` written to the tiled-dataset dir for the methodology's tile-count table.
- Wall-clock and peak GPU-memory logging per training stage.
- Vetted-image evaluation at multiple confidence thresholds (recall-first sweep).
- Plots: recall vs. confidence threshold, PR-curve overlay against earlier iterations,
  sunny/cloudy recall split.
- Single `thesis_manifest.json` collecting every measurement so the thesis tables and figures
  pull from one source of truth.

**Prerequisites:**
1. `yolo.zip` on Drive at `{BASE_DIR}/datasets/yolo.zip` with CVAT classes in canonical order.
2. A vetted-image set with YOLO-format labels at `{BASE_DIR}/inference/vetted/{images,labels}/`.
3. The pipeline package importable from `AEA_PIPELINE_ROOT` (defaults match the production
   notebook).

## Run config

In [ ]:
SMOKE_TEST = False
THESIS_RUN_ID = 'iter3_3class_plot_stratified'

# 'fly', 'butterfly', 'other'. Bumblebee excluded: 14 instances across the entire subset is
# below any reasonable supervised-training budget for a separate detector class.
KEEP_CLASSES = ['fly', 'butterfly', 'other']

# Plot-stratified split: each plot contributes proportionally to train/val/test. Set to False
# to consume whatever split is already in the source zip (legacy behaviour, image-level random
# shuffle, plot leakage between splits).
PLOT_STRATIFIED_SPLIT = True
VAL_FRAC = 0.20
TEST_FRAC = 0.10

## Install Colab deps

In [ ]:
import sys
if 'google.colab' in sys.modules:
    !pip install -q ultralytics sahi matplotlib

## 1. Train YOLO detector

Three-class, plot-stratified, with slicer stats and timing captured. Writes to
`{BASE_DIR}/runs/yolo/thesis/{THESIS_RUN_ID}/`.

In [ ]:
"""Train the thesis YOLO pollinator detector.

Same plumbing as the production pollinator_pipeline.ipynb, but:
  - 3 classes (fly, butterfly, other)
  - re-splits images per-plot before patching, so train/val/test each contain images from
    every plot in proportion to plot size (no plot leakage between splits)
  - writes slicer_stats.json and a per-stage training_stats.json to the run output dir
"""

import os
import sys
import json
import time
import shutil
import subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
BASE_DIR = os.environ.get(
    'AEA_BASE',
    '/content/drive/MyDrive/aea' if IN_COLAB else os.path.expanduser('~/aea'),
)
PIPELINE_ROOT = os.environ.get('AEA_PIPELINE_ROOT', f'{BASE_DIR}/ml-pipelines')
LOCAL_EXTRACT_DIR = os.environ.get(
    'AEA_EXTRACT',
    '/content/data' if IN_COLAB else f'{BASE_DIR}/extracted',
)

YOLO_ZIP_PATH = f'{BASE_DIR}/datasets/yolo.zip'
EXTRACTED_SUBDIR = 'yolo'
OUTPUT_DIR = f'{BASE_DIR}/runs/yolo/thesis/{THESIS_RUN_ID}'

# Class layout as exported by CVAT. Order must match the class indices in the label .txt files.
CVAT_CLASSES = ['bumblebee', 'fly', 'butterfly', 'other', 'unsure']

# Tile config - matches the methodology paragraph; do not change without also updating prose.
USE_TILES = True
TILE_SIZE = 640
TILE_OVERLAP = 0.2
TILE_MIN_AREA = 0.1
KEEP_EMPTY_TILES = {'train': 2, 'val': 5, 'test': True}

# Hyperparameters - match the methodology paragraph.
MODEL_SIZE = 'yolo26n.pt'
IMG_SIZE = TILE_SIZE if USE_TILES else 1024
BATCH = 32
EPOCHS_STAGE1 = 1 if SMOKE_TEST else 40
EPOCHS_STAGE2 = 1 if SMOKE_TEST else 70
LR_STAGE1 = 1e-3
LR_STAGE2 = 5e-4
SEED = 42


def ensure_extracted(zip_drive_path, extract_to, expected_subdir):
    target = Path(extract_to) / expected_subdir
    if target.exists() and any(target.iterdir()):
        print(f'[setup] reusing existing {target}')
        return str(target)
    target.mkdir(parents=True, exist_ok=True)
    local_zip = target.parent / Path(zip_drive_path).name
    print(f'[setup] copying {zip_drive_path} -> {local_zip}')
    shutil.copy(zip_drive_path, local_zip)
    print(f'[setup] unzipping into {target}')
    subprocess.run(['unzip', '-q', '-o', str(local_zip), '-d', str(target)], check=True)
    local_zip.unlink()
    nested = target / expected_subdir
    if nested.is_dir():
        for item in list(nested.iterdir()):
            shutil.move(str(item), str(target / item.name))
        nested.rmdir()
    print(f'[setup] ready at {target}')
    return str(target)



def _write_data_yaml(dataset_root, names):
    root = Path(dataset_root)
    lines = [f'path: {root}']
    for split in ('train', 'val', 'test'):
        if (root / 'images' / split).exists():
            lines.append(f'{split}: images/{split}')
    lines.append('names:')
    for i, name in enumerate(names):
        lines.append(f'  {i}: {name}')
    (root / 'data.yaml').write_text('\n'.join(lines) + '\n')
    print(f'[patch] rewrote {root / "data.yaml"} with {len(names)} classes')


def patch_dataset(dataset_root, cvat_classes, keep_classes, marker_id):
    root = Path(dataset_root)
    marker = root / '.patched_for'
    if marker.exists() and marker.read_text() == marker_id:
        print(f'[patch] dataset already patched for {marker_id}; skipping')
        _write_data_yaml(root, keep_classes)
        return
    cvat_to_idx = {name: i for i, name in enumerate(cvat_classes)}
    remap = {cvat_to_idx[n]: i for i, n in enumerate(keep_classes) if n in cvat_to_idx}
    n_lines_stripped = n_labels_removed = n_images_removed = 0
    for split in ('train', 'val', 'test'):
        labels_dir = root / 'labels' / split
        images_dir = root / 'images' / split
        if not labels_dir.exists():
            continue
        for label_file in labels_dir.glob('*.txt'):
            original = label_file.read_text().splitlines()
            kept = []
            for line in original:
                parts = line.strip().split()
                if not parts:
                    continue
                try:
                    cls = int(parts[0])
                except ValueError:
                    continue
                if cls not in remap:
                    continue
                parts[0] = str(remap[cls])
                kept.append(' '.join(parts))
            n_lines_stripped += len(original) - len(kept)
            if kept:
                label_file.write_text('\n'.join(kept) + '\n')
            else:
                label_file.unlink()
                n_labels_removed += 1
                if images_dir.exists():
                    for img in images_dir.glob(f'{label_file.stem}.*'):
                        img.unlink()
                        n_images_removed += 1
    print(
        f'[patch] keep={keep_classes}: stripped {n_lines_stripped} lines, removed '
        f'{n_labels_removed} empty labels and {n_images_removed} orphan images'
    )
    _write_data_yaml(root, keep_classes)
    marker.write_text(marker_id)


def ensure_tiled(source_root, extract_dir, tile_size, overlap, min_area, keep_empty,
                 source_config_id, classes, slice_seed):
    tiled_root = Path(extract_dir) / 'yolo-tiled'
    marker = tiled_root / '.sliced_for'
    empty_repr = (
        ','.join(f'{k}={v}' for k, v in sorted(keep_empty.items()))
        if isinstance(keep_empty, dict) else str(keep_empty)
    )
    slice_id = (
        f'source={source_config_id}|tile={tile_size}|overlap={overlap}'
        f'|min_area={min_area}|empty={empty_repr}|seed={slice_seed}'
    )
    if marker.exists() and marker.read_text() == slice_id:
        print(f'[tile] reusing cached tiled dataset at {tiled_root}')
        _write_data_yaml(tiled_root, classes)
        return str(tiled_root)
    if tiled_root.exists():
        print(f'[tile] tile config changed; rebuilding {tiled_root}')
        shutil.rmtree(tiled_root)
    from pollinator.training import slice_dataset
    print(
        f'[tile] slicing {source_root} -> {tiled_root} '
        f'(tile={tile_size}, overlap={overlap}, min_area={min_area}, empty={keep_empty})'
    )
    stats = slice_dataset(
        dataset_root=source_root,
        output_root=str(tiled_root),
        tile_size=tile_size,
        overlap=overlap,
        min_area=min_area,
        keep_empty_tiles=keep_empty,
        seed=slice_seed,
    )
    for split, s in stats.items():
        print(
            f'[tile] {split}: {s["source_images"]} src -> {s["tiles"]} tiles '
            f'({s["labeled_tiles"]} labeled)'
        )
    _write_data_yaml(tiled_root, classes)
    tiled_root.mkdir(parents=True, exist_ok=True)
    (tiled_root / 'slicer_stats.json').write_text(json.dumps(stats, indent=2))
    print(f'[tile] wrote {tiled_root / "slicer_stats.json"}')
    marker.write_text(slice_id)
    return str(tiled_root)


def _peak_gpu_mb():
    try:
        import torch
        if torch.cuda.is_available():
            return torch.cuda.max_memory_allocated() / 1024 / 1024
    except Exception:
        pass
    return None


def _reset_gpu_peak():
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
    except Exception:
        pass


def main_training():
    if IN_COLAB:
        from google.colab import drive
        drive.mount('/content/drive')
    if PIPELINE_ROOT and PIPELINE_ROOT not in sys.path:
        sys.path.insert(0, PIPELINE_ROOT)
    # Import after PIPELINE_ROOT is on sys.path.
    from pollinator.training import restratify_by_plot
    Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

    dataset_root = ensure_extracted(YOLO_ZIP_PATH, LOCAL_EXTRACT_DIR, EXTRACTED_SUBDIR)

    # Plot-stratified resplit BEFORE patching, so the patch markers reflect the new layout.
    if PLOT_STRATIFIED_SPLIT:
        stratified_id = f'val={VAL_FRAC}|test={TEST_FRAC}|seed={SEED}'
        restratify_by_plot(
            dataset_root, val_frac=VAL_FRAC, test_frac=TEST_FRAC,
            seed=SEED, marker_id=stratified_id,
        )

    source_config_id = (
        f"keep={','.join(KEEP_CLASSES)}|stratified={PLOT_STRATIFIED_SPLIT}"
        f"|val={VAL_FRAC}|test={TEST_FRAC}"
    )
    marker = Path(dataset_root) / '.patched_for'
    if marker.exists() and marker.read_text() != source_config_id:
        print('[setup] config changed since last run; re-extracting from zip...')
        shutil.rmtree(dataset_root)
        dataset_root = ensure_extracted(YOLO_ZIP_PATH, LOCAL_EXTRACT_DIR, EXTRACTED_SUBDIR)
        if PLOT_STRATIFIED_SPLIT:
            restratify_by_plot(
                dataset_root, val_frac=VAL_FRAC, test_frac=TEST_FRAC,
                seed=SEED, marker_id=stratified_id,
            )

    patch_dataset(
        dataset_root, cvat_classes=CVAT_CLASSES, keep_classes=KEEP_CLASSES,
        marker_id=source_config_id,
    )

    if USE_TILES:
        dataset_root = ensure_tiled(
            source_root=dataset_root, extract_dir=LOCAL_EXTRACT_DIR,
            tile_size=TILE_SIZE, overlap=TILE_OVERLAP, min_area=TILE_MIN_AREA,
            keep_empty=KEEP_EMPTY_TILES, source_config_id=source_config_id,
            classes=KEEP_CLASSES, slice_seed=SEED,
        )

    # Mirror the slicer_stats into the run output dir so the manifest can pick it up.
    src_stats = Path(dataset_root) / 'slicer_stats.json'
    if src_stats.exists():
        shutil.copy(src_stats, Path(OUTPUT_DIR) / 'slicer_stats.json')

    from pollinator.workflows.training_yolo import retrain_yolo

    timings = {}
    _reset_gpu_peak()
    t0 = time.perf_counter()
    result = retrain_yolo(
        use_tiles=False,  # tiling already done above
        dataset_root=dataset_root,
        output_dir=OUTPUT_DIR,
        model_size=MODEL_SIZE,
        img_size=IMG_SIZE,
        batch=BATCH,
        epochs_stage1=EPOCHS_STAGE1,
        epochs_stage2=EPOCHS_STAGE2,
        lr_stage1=LR_STAGE1,
        lr_stage2=LR_STAGE2,
        classes=KEEP_CLASSES,
        seed=SEED,
    )
    timings['wall_time_seconds'] = time.perf_counter() - t0
    timings['peak_gpu_mb'] = _peak_gpu_mb()

    training_stats = {
        'thesis_run_id': THESIS_RUN_ID,
        'classes': KEEP_CLASSES,
        'plot_stratified': PLOT_STRATIFIED_SPLIT,
        'config': {
            'model_size': MODEL_SIZE, 'img_size': IMG_SIZE, 'batch': BATCH,
            'epochs_stage1': EPOCHS_STAGE1, 'epochs_stage2': EPOCHS_STAGE2,
            'lr_stage1': LR_STAGE1, 'lr_stage2': LR_STAGE2,
            'tile_size': TILE_SIZE, 'tile_overlap': TILE_OVERLAP,
            'tile_min_area': TILE_MIN_AREA, 'keep_empty_tiles': KEEP_EMPTY_TILES,
            'seed': SEED,
        },
        'timings': timings,
        'result': result,
    }
    (Path(OUTPUT_DIR) / 'training_stats.json').write_text(
        json.dumps(training_stats, indent=2, default=str)
    )
    print(f'\n=== Training complete ===')
    print(f'wall_time = {timings["wall_time_seconds"]/60:.1f} min')
    print(f'peak_gpu  = {timings["peak_gpu_mb"]} MB' if timings['peak_gpu_mb'] else 'peak_gpu  = n/a')
    return training_stats


training_stats = main_training()

## 2. Vetted-image inference

Run inference on the pinned vetted-image set at a low confidence (0.05) so the prediction
dump can be post-filtered to any threshold without re-running YOLO. Outputs
`vetted_predictions.json` in the run dir.

The vetted set must live at `{BASE_DIR}/inference/vetted/images/*.jpg` with matching YOLO
labels at `{BASE_DIR}/inference/vetted/labels/*.txt`.

In [ ]:
VETTED_DIR = f'{BASE_DIR}/inference/vetted'
VETTED_LOW_CONF = 0.05  # store everything; we filter higher thresholds in software later.

from pollinator.detection.yolo_detector import YoloDetector  # noqa: E402

checkpoint = f'{OUTPUT_DIR}/stage2_finetune/weights/best.pt'
vetted_images = sorted(Path(f'{VETTED_DIR}/images').glob('*.[jJ][pP][gG]')) \
    + sorted(Path(f'{VETTED_DIR}/images').glob('*.png'))
print(f'[vetted] {len(vetted_images)} images in {VETTED_DIR}/images')

detector = YoloDetector(
    checkpoint_path=checkpoint,
    use_sahi=True,
    slice_size=TILE_SIZE,
    overlap=TILE_OVERLAP,
    confidence=VETTED_LOW_CONF,
    iou=0.5,
)

vetted_predictions = {}
t0 = time.perf_counter()
for img_path in vetted_images:
    dets = detector.predict(img_path)
    vetted_predictions[img_path.name] = dets
vetted_inference_seconds = time.perf_counter() - t0

out_path = Path(OUTPUT_DIR) / 'vetted_predictions.json'
out_path.write_text(json.dumps({
    'checkpoint': checkpoint,
    'low_conf_threshold': VETTED_LOW_CONF,
    'image_count': len(vetted_images),
    'wall_time_seconds': vetted_inference_seconds,
    'per_image_seconds': vetted_inference_seconds / max(1, len(vetted_images)),
    'predictions': vetted_predictions,
}, indent=2, default=str))
print(f'[vetted] wrote {out_path}')
print(f'[vetted] inference took {vetted_inference_seconds/60:.2f} min '
      f'({vetted_inference_seconds/max(1, len(vetted_images)):.2f} s/image)')

## 3. Evaluation sweep

Greedy IoU matching at IoU\,$\geq$\,0.5, per-class TP/FP/FN at six confidence thresholds.
Outputs `evaluation_sweep.json` with per-threshold per-class precision/recall/F1.

In [ ]:
from PIL import Image

CONF_THRESHOLDS = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
IOU_THRESHOLD = 0.5

# Classes to report in the sweep printout and the plots. Keep this a subset of
# KEEP_CLASSES. Narrow it to just the classes that have meaningful GT coverage
# in the vetted set: e.g. ['fly'] when the vetted set is fly-dominated and the
# other classes would only contribute zero-GT rows that make the model look
# worse than it is.
REPORT_CLASSES = ['fly']


def yolo_label_to_xyxy(line, img_w, img_h):
    parts = line.strip().split()
    cls_idx = int(parts[0])
    cx, cy, w, h = (float(x) for x in parts[1:5])
    x1 = (cx - w / 2) * img_w
    y1 = (cy - h / 2) * img_h
    x2 = (cx + w / 2) * img_w
    y2 = (cy + h / 2) * img_h
    return cls_idx, [x1, y1, x2, y2]


def load_ground_truth(vetted_dir, cvat_classes, keep_classes):
    """Read YOLO .txt labels written in the CVAT index space and return GT
    boxes per image with class names restricted to keep_classes.

    The vetted labels store class indices in CVAT's original ordering
    (CVAT_CLASSES). The trained model uses the patched KEEP_CLASSES ordering.
    Reading CVAT indices through KEEP_CLASSES would silently mislabel every
    box; remapping by name keeps them aligned with what the model predicts.
    """
    keep_set = set(keep_classes)
    gt = {}
    images_dir = Path(vetted_dir) / 'images'
    labels_dir = Path(vetted_dir) / 'labels'
    for img in sorted(images_dir.iterdir()):
        if img.suffix.lower() not in {'.jpg', '.jpeg', '.png'}:
            continue
        lbl = labels_dir / (img.stem + '.txt')
        with Image.open(img) as im:
            w, h = im.size
        items = []
        if lbl.exists():
            for line in lbl.read_text().splitlines():
                if not line.strip():
                    continue
                idx, box = yolo_label_to_xyxy(line, w, h)
                if not (0 <= idx < len(cvat_classes)):
                    continue
                name = cvat_classes[idx]
                if name in keep_set:
                    items.append((name, box))
        gt[img.name] = items
    return gt


def iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    if ix2 <= ix1 or iy2 <= iy1:
        return 0.0
    inter = (ix2 - ix1) * (iy2 - iy1)
    aa = (a[2] - a[0]) * (a[3] - a[1])
    bb = (b[2] - b[0]) * (b[3] - b[1])
    return inter / (aa + bb - inter)


def evaluate_at_threshold(preds_per_image, gt_per_image, conf_t, iou_t, classes):
    stats = {c: {'tp': 0, 'fp': 0, 'fn': 0} for c in classes}
    for img_name, preds in preds_per_image.items():
        filtered = sorted(
            [p for p in preds if p.get('confidence', 0) >= conf_t],
            key=lambda p: -p.get('confidence', 0),
        )
        gts = list(gt_per_image.get(img_name, []))
        matched = set()
        for p in filtered:
            cls = p.get('class')
            pb = [p['x1'], p['y1'], p['x2'], p['y2']]
            best, best_i = 0.0, None
            for i, (gc, gb) in enumerate(gts):
                if i in matched or gc != cls:
                    continue
                v = iou(pb, gb)
                if v > best:
                    best, best_i = v, i
            if best_i is not None and best >= iou_t:
                stats[cls]['tp'] += 1
                matched.add(best_i)
            elif cls in stats:
                stats[cls]['fp'] += 1
        for i, (gc, _) in enumerate(gts):
            if i not in matched and gc in stats:
                stats[gc]['fn'] += 1
    out = {}
    for c, s in stats.items():
        tp, fp, fn = s['tp'], s['fp'], s['fn']
        p = tp / max(1, tp + fp)
        r = tp / max(1, tp + fn)
        f1 = 2 * p * r / max(1e-9, p + r)
        out[c] = {'tp': tp, 'fp': fp, 'fn': fn, 'precision': p, 'recall': r, 'f1': f1}
    return out


vetted_dump = json.loads((Path(OUTPUT_DIR) / 'vetted_predictions.json').read_text())
preds_per_image = vetted_dump['predictions']
gt_per_image = load_ground_truth(VETTED_DIR, CVAT_CLASSES, KEEP_CLASSES)

# Per-class GT count audit so a low recall for any class can be traced to
# either a model issue or a dataset gap.
from collections import Counter as _Counter
gt_counts = _Counter(gc for items in gt_per_image.values() for gc, _ in items)
print(f'[eval] vetted GT counts (after CVAT->KEEP remap): {dict(gt_counts)}')
for c in REPORT_CLASSES:
    if gt_counts.get(c, 0) == 0:
        print(f'[eval] WARNING: reporting "{c}" but vetted set has 0 GT for it')

sweep = {}
for t in CONF_THRESHOLDS:
    sweep[str(t)] = evaluate_at_threshold(
        preds_per_image, gt_per_image, t, IOU_THRESHOLD, KEEP_CLASSES,
    )

sweep_path = Path(OUTPUT_DIR) / 'evaluation_sweep.json'
sweep_path.write_text(json.dumps({
    'classes_trained': KEEP_CLASSES,
    'classes_reported': REPORT_CLASSES,
    'cvat_classes': CVAT_CLASSES,
    'conf_thresholds': CONF_THRESHOLDS,
    'iou_threshold': IOU_THRESHOLD,
    'image_count': len(gt_per_image),
    'gt_counts': dict(gt_counts),
    'per_threshold': sweep,
}, indent=2))
print(f'[eval] wrote {sweep_path}')
for t in CONF_THRESHOLDS:
    line = f'  conf={t:.2f}'
    for c in REPORT_CLASSES:
        m = sweep[str(t)][c]
        line += f'  {c}: R={m["recall"]:.3f} P={m["precision"]:.3f} F1={m["f1"]:.3f}'
    print(line)

## 4. Plots

Recall vs. confidence threshold per class, plus a PR-curve overlay against any earlier
iteration checkpoints found under `{BASE_DIR}/runs/yolo/` (Iteration 1, Iteration 2, etc).
Skips gracefully if earlier checkpoints are not present.

In [ ]:
import matplotlib.pyplot as plt

FIG_DIR = Path(OUTPUT_DIR) / 'figures'
FIG_DIR.mkdir(exist_ok=True)

# ---- (a) recall vs confidence threshold ----
fig, ax = plt.subplots(figsize=(7, 4))
for c in REPORT_CLASSES:
    rs = [sweep[str(t)][c]['recall'] for t in CONF_THRESHOLDS]
    ax.plot(CONF_THRESHOLDS, rs, marker='o', label=c)
ax.set_xlabel('Confidence threshold')
ax.set_ylabel('Recall')
ax.set_title(f'Recall vs. confidence threshold ({THESIS_RUN_ID})')
ax.set_ylim(0, 1.02)
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / 'recall_vs_threshold.png', dpi=150)
plt.show()

# ---- (b) PR curve from the threshold sweep (per class) ----
fig, ax = plt.subplots(figsize=(7, 4))
for c in REPORT_CLASSES:
    ps = [sweep[str(t)][c]['precision'] for t in CONF_THRESHOLDS]
    rs = [sweep[str(t)][c]['recall'] for t in CONF_THRESHOLDS]
    ax.plot(rs, ps, marker='o', label=c)
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title(f'Precision-Recall (six thresholds; {THESIS_RUN_ID})')
ax.set_xlim(0, 1.02)
ax.set_ylim(0, 1.02)
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / 'pr_curve_sweep.png', dpi=150)
plt.show()

print(f'[plots] wrote figures to {FIG_DIR}')
print('Multi-iteration PR overlay is left as a follow-up: run this notebook for each')
print('iteration checkpoint, then collect the per_threshold blobs from evaluation_sweep.json')
print('and overlay them in a separate figure.')

## 5. Thesis information

Consolidates every measurement into one JSON file

In [ ]:
def _load_or_none(path):
    p = Path(path)
    return json.loads(p.read_text()) if p.exists() else None


# Manifest is re-runnable on its own: pull from in-memory vars when present,
# fall back to the on-disk JSON the upstream cells wrote. Avoids NameError
# when the kernel was restarted between training and writing the manifest.
training_stats_disk = _load_or_none(Path(OUTPUT_DIR) / 'training_stats.json')
training_stats_obj = locals().get('training_stats') or training_stats_disk

vetted_dump_disk = _load_or_none(Path(OUTPUT_DIR) / 'vetted_predictions.json') or {}
vetted_timing = {
    'wall_time_seconds': locals().get('vetted_inference_seconds')
        or vetted_dump_disk.get('wall_time_seconds'),
    'per_image_seconds': vetted_dump_disk.get('per_image_seconds'),
    'image_count': (
        len(locals()['vetted_images']) if 'vetted_images' in locals()
        else vetted_dump_disk.get('image_count')
    ),
}

manifest = {
    'thesis_run_id': THESIS_RUN_ID,
    'classes': KEEP_CLASSES,
    'plot_stratified': PLOT_STRATIFIED_SPLIT,
    'training_stats_file': str(Path(OUTPUT_DIR) / 'training_stats.json'),
    'slicer_stats_file': str(Path(OUTPUT_DIR) / 'slicer_stats.json'),
    'vetted_predictions_file': str(Path(OUTPUT_DIR) / 'vetted_predictions.json'),
    'evaluation_sweep_file': str(Path(OUTPUT_DIR) / 'evaluation_sweep.json'),
    'figures_dir': str(Path(OUTPUT_DIR) / 'figures'),
    'embedded': {
        'slicer_stats': _load_or_none(Path(OUTPUT_DIR) / 'slicer_stats.json'),
        'training_stats': training_stats_obj,
        'evaluation_sweep': _load_or_none(Path(OUTPUT_DIR) / 'evaluation_sweep.json'),
        'vetted_inference_timing': vetted_timing,
    },
}
manifest_path = Path(OUTPUT_DIR) / 'thesis_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2, default=str))
print(f'[manifest] wrote {manifest_path}')
print(f'[manifest] thesis section can now pull every measurement from one file.')